# 03 - Hybrid GraphRAG: Vector Search + Graph Traversal

**Goal**: Build a "true" GraphRAG system that uses actual graph traversal at query time.

**Approach B**: Vector search finds seed entities -> Graph traversal expands context -> LLM generates answer

**Comparison**: We'll compare this hybrid approach with Microsoft GraphRAG's Local Search.

**Prerequisites**:
- Neo4j running (`docker-compose up -d`)
- `graphrag index` completed (notebook 02)
- OpenAI API key in `.env`

**Time**: ~30 minutes

## Step 1: Load GraphRAG Data into Neo4j

We'll take the entities and relationships that GraphRAG extracted (from parquet files)
and load them into Neo4j as a proper knowledge graph.

In [ ]:
import os
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI

load_dotenv()

# Neo4j connection
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASS = os.getenv("NEO4J_PASSWORD", "graphrag-password")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
driver.verify_connectivity()

# OpenAI client
openai_client = OpenAI()

# Load GraphRAG output
OUTPUT_DIR = Path("..") / "graphrag_workspace" / "output"
entities_df = pd.read_parquet(OUTPUT_DIR / "entities.parquet")
rels_df = pd.read_parquet(OUTPUT_DIR / "relationships.parquet")
text_units_df = pd.read_parquet(OUTPUT_DIR / "text_units.parquet")

print(f"Entities: {len(entities_df)}, Relationships: {len(rels_df)}, Text chunks: {len(text_units_df)}")
print("Connected to Neo4j and OpenAI!")

In [ ]:
def run_query(query, parameters=None):
    """Run a Cypher query and return results."""
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return [record.data() for record in result]

# Clear existing data
run_query("MATCH (n) DETACH DELETE n")

# Drop existing indexes (ignore errors if they don't exist)
try:
    run_query("DROP INDEX entity_embedding IF EXISTS")
except:
    pass

print("Database cleared.")

In [ ]:
# Create entities as nodes in Neo4j
for _, entity in entities_df.iterrows():
    run_query("""
        CREATE (e:Entity {
            id: $id,
            name: $name,
            type: $type,
            description: $description,
            text_unit_ids: $text_unit_ids
        })
    """, {
        "id": entity["id"],
        "name": entity["title"],
        "type": entity["type"],
        "description": entity["description"] or "",
        "text_unit_ids": list(entity["text_unit_ids"]) if entity["text_unit_ids"] is not None else []
    })

print(f"Created {len(entities_df)} entity nodes.")

In [ ]:
# Create relationships as edges
created = 0
for _, rel in rels_df.iterrows():
    result = run_query("""
        MATCH (s:Entity {name: $source}), (t:Entity {name: $target})
        CREATE (s)-[r:RELATED_TO {
            id: $id,
            description: $description,
            weight: $weight
        }]->(t)
        RETURN count(r) AS cnt
    """, {
        "id": rel["id"],
        "source": rel["source"],
        "target": rel["target"],
        "description": rel["description"] or "",
        "weight": float(rel["weight"])
    })
    created += result[0]["cnt"] if result else 0

print(f"Created {created} relationships.")

In [ ]:
# Store text chunks as nodes too (for retrieval later)
for _, unit in text_units_df.iterrows():
    run_query("""
        CREATE (t:TextChunk {
            id: $id,
            text: $text,
            n_tokens: $n_tokens
        })
    """, {
        "id": unit["id"],
        "text": unit["text"],
        "n_tokens": int(unit["n_tokens"])
    })

# Link entities to their source text chunks
linked = 0
for _, entity in entities_df.iterrows():
    if entity["text_unit_ids"] is not None:
        for tu_id in entity["text_unit_ids"]:
            result = run_query("""
                MATCH (e:Entity {id: $entity_id}), (t:TextChunk {id: $chunk_id})
                CREATE (e)-[:MENTIONED_IN]->(t)
                RETURN count(*) AS cnt
            """, {"entity_id": entity["id"], "chunk_id": tu_id})
            linked += result[0]["cnt"] if result else 0

print(f"Created {len(text_units_df)} text chunk nodes.")
print(f"Linked {linked} entity-to-chunk connections.")

In [ ]:
# Verify the graph
stats = run_query("""
    MATCH (e:Entity) WITH count(e) AS entities
    MATCH ()-[r:RELATED_TO]->() WITH entities, count(r) AS relationships
    MATCH (t:TextChunk) WITH entities, relationships, count(t) AS chunks
    RETURN entities, relationships, chunks
""")
print(f"Graph loaded: {stats[0]}")
print()

# Show entity types
types = run_query("""
    MATCH (e:Entity)
    RETURN e.type AS type, count(*) AS count
    ORDER BY count DESC
""")
print("Entity types:")
for t in types:
    print(f"  {t['type']}: {t['count']}")

## Step 2: Add Vector Index to Neo4j

Neo4j supports native vector indexes. We'll embed each entity's description
and store it directly in the graph — so vector search and graph traversal happen in the same database.

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    """Get embedding vector from OpenAI."""
    response = openai_client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

# Embed all entity descriptions and store in Neo4j
entities_with_desc = entities_df[entities_df["description"].notna() & (entities_df["description"] != "")]

print(f"Embedding {len(entities_with_desc)} entity descriptions...")

for _, entity in entities_with_desc.iterrows():
    # Combine name + description for better embedding
    text_to_embed = f"{entity['title']}: {entity['description']}"
    embedding = get_embedding(text_to_embed)
    
    run_query("""
        MATCH (e:Entity {id: $id})
        SET e.embedding = $embedding
    """, {"id": entity["id"], "embedding": embedding})

print("Embeddings stored in Neo4j!")

In [ ]:
# Create vector index in Neo4j
# text-embedding-3-small produces 1536-dimensional vectors
run_query("""
    CREATE VECTOR INDEX entity_embedding IF NOT EXISTS
    FOR (e:Entity)
    ON (e.embedding)
    OPTIONS {
        indexConfig: {
            `vector.dimensions`: 1536,
            `vector.similarity_function`: 'cosine'
        }
    }
""")

print("Vector index created!")
print("Now Neo4j can do both: vector search AND graph traversal.")

## Step 3: Build the Hybrid Retriever

This is the core of our approach:
1. **Vector search** — find seed entities similar to the question
2. **Graph traversal** — expand from seeds by following relationships
3. **Text chunk collection** — gather source text for grounding
4. **Context formatting** — structure everything for the LLM

In [ ]:
def vector_search_entities(question, top_k=5):
    """Step 1: Find seed entities via vector similarity."""
    question_embedding = get_embedding(question)
    
    results = run_query("""
        CALL db.index.vector.queryNodes('entity_embedding', $top_k, $embedding)
        YIELD node, score
        RETURN node.name AS name, node.type AS type, 
               node.description AS description, score
        ORDER BY score DESC
    """, {"top_k": top_k, "embedding": question_embedding})
    
    return results

# Test it
seeds = vector_search_entities("What did Geoffrey Hinton contribute to deep learning?")
print("Seed entities found:")
for s in seeds:
    print(f"  [{s['type']}] {s['name']} (score: {s['score']:.3f})")

In [ ]:
def graph_traverse(seed_names, max_hops=2):
    """Step 2: Expand from seed entities by traversing the graph.
    
    This is where the graph structure truly shines — we follow
    relationships to discover connected entities that vector search alone
    would miss.
    """
    results = run_query("""
        // Start from seed entities
        MATCH (seed:Entity)
        WHERE seed.name IN $seed_names
        
        // Traverse up to N hops
        MATCH path = (seed)-[r:RELATED_TO*1..""" + str(max_hops) + """]->(connected:Entity)
        
        // Collect the full subgraph
        UNWIND relationships(path) AS rel
        WITH DISTINCT startNode(rel) AS from_node, rel, endNode(rel) AS to_node
        
        RETURN from_node.name AS source,
               from_node.type AS source_type,
               rel.description AS relationship,
               rel.weight AS weight,
               to_node.name AS target,
               to_node.type AS target_type
        ORDER BY weight DESC
    """, {"seed_names": seed_names})
    
    return results

# Test: traverse from top 3 seed entities
seed_names = [s["name"] for s in seeds[:3]]
print(f"Traversing from seeds: {seed_names}")
print(f"Max hops: 2\n")

subgraph = graph_traverse(seed_names, max_hops=2)
print(f"Found {len(subgraph)} relationships in subgraph:")
for edge in subgraph[:10]:
    print(f"  [{edge['source_type']}] {edge['source']}")
    print(f"    --({edge['weight']:.0f})--> [{edge['target_type']}] {edge['target']}")
    print(f"    {edge['relationship'][:80]}...")
    print()

In [ ]:
def collect_text_chunks(seed_names):
    """Step 3: Get source text chunks linked to discovered entities."""
    results = run_query("""
        MATCH (e:Entity)-[:MENTIONED_IN]->(t:TextChunk)
        WHERE e.name IN $names
        RETURN DISTINCT t.text AS text, t.n_tokens AS tokens,
               collect(DISTINCT e.name) AS mentioned_entities
    """, {"names": seed_names})
    
    return results

# Collect all entity names from the subgraph
all_entity_names = set(seed_names)
for edge in subgraph:
    all_entity_names.add(edge["source"])
    all_entity_names.add(edge["target"])

chunks = collect_text_chunks(list(all_entity_names))
print(f"Found {len(chunks)} text chunks from {len(all_entity_names)} entities")
for c in chunks:
    print(f"  Chunk ({c['tokens']} tokens), mentions: {c['mentioned_entities'][:5]}...")

## Step 4: Format Context for LLM

This is a key design decision: how do we present the subgraph to the LLM?

We'll use a **layered format**: structured graph data + natural language descriptions + source text.

In [ ]:
def format_context(seed_entities, subgraph_edges, text_chunks):
    """Step 4: Format the retrieved context for the LLM.
    
    Layers:
    1. Seed entities with descriptions (most relevant)
    2. Subgraph relationships (structural context)
    3. Source text chunks (grounding)
    """
    context_parts = []
    
    # Layer 1: Seed entities
    context_parts.append("=== Key Entities ===")
    for entity in seed_entities:
        context_parts.append(
            f"- {entity['name']} [{entity['type']}] (relevance: {entity['score']:.2f}): "
            f"{entity['description']}"
        )
    
    # Layer 2: Relationships from graph traversal
    context_parts.append("\n=== Relationships (from graph traversal) ===")
    for edge in subgraph_edges:
        context_parts.append(
            f"- {edge['source']} -> {edge['target']}: {edge['relationship']}"
        )
    
    # Layer 3: Source text
    context_parts.append("\n=== Source Text ===")
    for chunk in text_chunks:
        context_parts.append(chunk["text"])
    
    return "\n".join(context_parts)

# Build context
context = format_context(seeds, subgraph, chunks)
print(f"Context length: {len(context)} chars")
print("\n--- Context Preview (first 1000 chars) ---")
print(context[:1000])

## Step 5: Generate Answer with LLM

Now we send the structured context to the LLM to generate an answer.

In [ ]:
def hybrid_graphrag_query(question, top_k_seeds=5, max_hops=2):
    """Full hybrid GraphRAG pipeline: vector search + graph traversal + LLM."""
    
    # Step 1: Vector search for seed entities
    seeds = vector_search_entities(question, top_k=top_k_seeds)
    seed_names = [s["name"] for s in seeds]
    
    # Step 2: Graph traversal from seeds
    subgraph = graph_traverse(seed_names, max_hops=max_hops)
    
    # Step 3: Collect text chunks
    all_names = set(seed_names)
    for edge in subgraph:
        all_names.add(edge["source"])
        all_names.add(edge["target"])
    chunks = collect_text_chunks(list(all_names))
    
    # Step 4: Format context
    context = format_context(seeds, subgraph, chunks)
    
    # Step 5: LLM generation
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": (
                "You are a helpful assistant. Answer the question based on the provided context. "
                "The context includes: key entities found via vector search, relationships discovered "
                "via graph traversal, and source text. Use all layers to provide a comprehensive answer. "
                "If the context doesn't contain enough information, say so."
            )},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        temperature=0
    )
    
    return {
        "answer": response.choices[0].message.content,
        "seeds": seeds,
        "subgraph_edges": len(subgraph),
        "entities_discovered": len(all_names),
        "text_chunks": len(chunks),
        "context_length": len(context)
    }

In [ ]:
# Test query 1: Entity-focused (same as MS GraphRAG test)
question1 = "What did Geoffrey Hinton contribute to deep learning?"
result1 = hybrid_graphrag_query(question1)

print(f"Question: {question1}")
print(f"\nRetrieval stats:")
print(f"  Seed entities: {[s['name'] for s in result1['seeds']]}")
print(f"  Graph edges traversed: {result1['subgraph_edges']}")
print(f"  Total entities discovered: {result1['entities_discovered']}")
print(f"  Text chunks retrieved: {result1['text_chunks']}")
print(f"  Context length: {result1['context_length']} chars")
print(f"\n{'='*60}")
print(f"Answer:\n{result1['answer']}")

In [ ]:
# Test query 2: Multi-hop question (where graph traversal really helps)
question2 = "How are Geoffrey Hinton and the Transformer architecture connected?"
result2 = hybrid_graphrag_query(question2)

print(f"Question: {question2}")
print(f"\nRetrieval stats:")
print(f"  Seed entities: {[s['name'] for s in result2['seeds']]}")
print(f"  Graph edges traversed: {result2['subgraph_edges']}")
print(f"  Total entities discovered: {result2['entities_discovered']}")
print(f"\n{'='*60}")
print(f"Answer:\n{result2['answer']}")

In [ ]:
# Test query 3: Relationship exploration
question3 = "What is the relationship between Ian Goodfellow and Yoshua Bengio?"
result3 = hybrid_graphrag_query(question3)

print(f"Question: {question3}")
print(f"\nRetrieval stats:")
print(f"  Seed entities: {[s['name'] for s in result3['seeds']]}")
print(f"  Graph edges traversed: {result3['subgraph_edges']}")
print(f"  Total entities discovered: {result3['entities_discovered']}")
print(f"\n{'='*60}")
print(f"Answer:\n{result3['answer']}")

## Step 6: The Power of Graph Traversal — Cypher Queries

Unlike MS GraphRAG, we can now run **precise structural queries** that are impossible
with vector search alone.

In [ ]:
# Query: Shortest path between two entities
print("=== Shortest path: Hinton <-> Transformer ===")
paths = run_query("""
    MATCH path = shortestPath(
        (a:Entity {name: 'GEOFFREY HINTON'})-[:RELATED_TO*]-(b:Entity {name: 'TRANSFORMER ARCHITECTURE'})
    )
    RETURN [n IN nodes(path) | n.name] AS entities,
           [r IN relationships(path) | r.description] AS relationships,
           length(path) AS hops
""")
if paths:
    p = paths[0]
    print(f"  Hops: {p['hops']}")
    print(f"  Path: {' -> '.join(p['entities'])}")
    for i, rel in enumerate(p['relationships']):
        print(f"  Step {i+1}: {rel[:100]}")
else:
    print("  No path found.")

In [ ]:
# Query: Find all people connected within 2 hops of Hinton
print("=== People within 2 hops of Hinton ===")
people = run_query("""
    MATCH path = (h:Entity {name: 'GEOFFREY HINTON'})-[:RELATED_TO*1..2]-(p:Entity {type: 'PERSON'})
    WHERE p.name <> 'GEOFFREY HINTON'
    RETURN DISTINCT p.name AS person, length(path) AS distance
    ORDER BY distance
""")
for p in people:
    print(f"  {p['person']} ({p['distance']} hop{'s' if p['distance'] > 1 else ''})")

In [ ]:
# Query: Most connected entities (hubs)
print("=== Most connected entities (hubs) ===")
hubs = run_query("""
    MATCH (e:Entity)-[r:RELATED_TO]-()
    RETURN e.name AS entity, e.type AS type, count(r) AS connections
    ORDER BY connections DESC
    LIMIT 10
""")
for h in hubs:
    print(f"  {h['entity']} [{h['type']}]: {h['connections']} connections")

## Step 7: Comparison — Hybrid vs MS GraphRAG

Let's compare the two approaches side by side.

In [ ]:
print("""Comparison: Hybrid GraphRAG vs MS GraphRAG
{'='*60}

MS GraphRAG (Local Search):
  - Retrieval: Vector similarity on entities -> table lookup
  - No actual graph traversal at query time
  - Cannot do: shortest path, variable-depth search, structural queries
  - Good at: broad context retrieval, theme discovery (Global Search)

Hybrid GraphRAG (this notebook):
  - Retrieval: Vector search (seeds) + Cypher graph traversal
  - Real graph traversal: shortest path, N-hop expansion, structural queries
  - Can combine: "find similar entities" + "follow their relationships"
  - Trade-off: needs Neo4j running, more setup

When to use which:
  - MS GraphRAG: Quick setup, document summarization, theme analysis
  - Hybrid: Precise relationship queries, multi-hop reasoning, structural analysis
  - Best: Combine both — MS GraphRAG for indexing/Global Search,
          Neo4j for precise queries at runtime
""")

In [ ]:
# Cleanup
driver.close()
print("Done! Neo4j connection closed.")